# Compare Darija Whisper Models (Kaggle T4 GPU)

Benchmark 3 HuggingFace Darija ASR models on the same audio clip and pick the
best one for your use case.

| # | Model | Type | Params | Reported WER |
|---|-------|------|--------|-------------|
| A | `Ayoub-Laachir/MaghrebVoice` | Full fine-tune (large-v3) | 1.55B | 3.15% |
| B | `Marialab/finetuned-whisper-large-v3-5000-step` | Full fine-tune (large-v3) | 1.55B | BLEU 0.74 |
| C | `anaszil/whisper-large-v3-turbo-darija` | LoRA adapter (turbo) | 809M + 50M | 24.9% |

**Setup:**  
- Settings → Accelerator → **GPU T4 x2** (or P100, A100)  
- Internet **On** (first run downloads ~6–15 GB of model weights)  
- Runtime → **Run all** (or step through cell by cell)

## 1. Install dependencies

In [ ]:
!pip -q install "transformers>=4.47" "torch>=2.0" "accelerate" "librosa" "soundfile" "peft" "jiwer" "tabulate"


## 2. Get a sample audio file

Option A — Upload an `.mp3` / `.wav` via Kaggle's **Add Data** → Upload, then
set `SAMPLE_PATH` to its path.

Option B — Download from a public URL (paste the URL below).  

Option C — Use a pre-loaded Kaggle dataset (e.g. your broadcast dataset).

Edit `SAMPLE_PATH` in the next cell after uploading.

In [ ]:
import os, torch, time, json, gc
import librosa
import soundfile as sf
from pathlib import Path

# --- CONFIGURE THESE ---
SAMPLE_PATH = None          # set after uploading, or leave None for URL
SAMPLE_URL  = None          # e.g. "https://example.com/sample.mp3"
OUT_DIR     = Path("/kaggle/working/compare")
# -----------------------

OUT_DIR.mkdir(parents=True, exist_ok=True)
WAV_PATH = OUT_DIR / "sample_16khz.wav"

if SAMPLE_PATH and os.path.exists(SAMPLE_PATH):
    src = SAMPLE_PATH
elif SAMPLE_URL:
    src = str(OUT_DIR / "sample_src")
    !wget -q "{SAMPLE_URL}" -O "{src}"
else:
    raise RuntimeError(
        "Set SAMPLE_PATH (after upload) or SAMPLE_URL in the cell above."
    )

audio, sr = librosa.load(src, sr=16000, mono=True)
sf.write(str(WAV_PATH), audio, 16000)
duration = len(audio) / 16000
print(f"Audio: {WAV_PATH}  ({duration:.1f}s, {sr} Hz, mono)")

## 3. Benchmark helper

The function below loads a model, transcribes the sample, measures time, and
cleans up GPU memory so the next model starts fresh.

In [ ]:
import sys


def transcribe_and_time(
    model_id: str,
    audio_path: str,
    is_lora: bool = False,
    lora_base: str = "openai/whisper-large-v3-turbo",
) -> dict:
    """Load *model_id*, transcribe *audio_path*, return timing + text."""
    device = 0 if torch.cuda.is_available() else "cpu"
    dtype  = torch.float16 if torch.cuda.is_available() else torch.float32
    result = {"model": model_id, "load_s": 0, "transcribe_s": 0, "text": ""}

    # ---- Load ----
    t0 = time.time()
    base = None  # declared here so cleanup is always safe
    model = None
    try:
        if is_lora:
            from peft import PeftModel
            from transformers import (
                WhisperForConditionalGeneration, WhisperProcessor, pipeline,
            )
            base = WhisperForConditionalGeneration.from_pretrained(
                lora_base, torch_dtype=dtype, low_cpu_mem_usage=True,
            )
            model = PeftModel.from_pretrained(base, model_id)
            processor = WhisperProcessor.from_pretrained(
                lora_base, language="Arabic", task="transcribe",
            )
            pipe = pipeline(
                "automatic-speech-recognition",
                model=model,
                tokenizer=processor.tokenizer,
                feature_extractor=processor.feature_extractor,
                chunk_length_s=30,
                device=device,
            )
        else:
            from transformers import pipeline as pl
            pipe = pl(
                "automatic-speech-recognition",
                model=model_id,
                device=device,
            )
    except Exception as e:
        result["error"] = str(e)
        return result
    result["load_s"] = round(time.time() - t0, 1)

    # ---- Transcribe ----
    t0 = time.time()
    try:
        out = pipe(audio_path, return_timestamps=True)
        result["text"] = out["text"].strip()
        result["chunks"] = out.get("chunks", [])
    except Exception as e:
        result["error"] = str(e)
        result["text"] = "[TRANSCRIPTION FAILED]"
    result["transcribe_s"] = round(time.time() - t0, 1)

    # ---- Cleanup (guard each name — non-LoRA path never sets model/base) ----
    del pipe
    if model is not None:
        del model
    if base is not None:
        del base
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result


def pretty_print(result: dict):
    """Print a single model's result to stderr (keeps notebook clean)."""
    tag = result["model"].split("/")[-1]
    err = result.get("error")
    print(f"\n{'='*60}", file=sys.stderr)
    print(f"  {tag}", file=sys.stderr)
    print(f"  Load: {result['load_s']}s  |  Transcribe: {result['transcribe_s']}s", file=sys.stderr)
    if err:
        print(f"  ERROR: {err}", file=sys.stderr)
    print(f"{'='*60}", file=sys.stderr)
    print(result["text"][:500], "\n..." if len(result["text"]) > 500 else "")


## 4. Run benchmarks

Each cell runs one model.  If you only want to compare 2 out of 3, skip the
cells you don't need (the comparison table will show `N/A` for skipped ones).

---
### Model A — `Ayoub-Laachir/MaghrebVoice`  
Full large-v3 fine-tune.  Best-reported WER (3.15%).  Heavy download (~6 GB).

In [ ]:
result_a = transcribe_and_time(
    "Ayoub-Laachir/MaghrebVoice", str(WAV_PATH), is_lora=False,
)
pretty_print(result_a)

### Model B — `Marialab/finetuned-whisper-large-v3-5000-step`  
Full large-v3 fine-tune on Darija-C.  Evaluated with BLEU (0.744).  Heavy download (~6 GB).

In [ ]:
result_b = transcribe_and_time(
    "Marialab/finetuned-whisper-large-v3-5000-step", str(WAV_PATH), is_lora=False,
)
pretty_print(result_b)

### Model C — `anaszil/whisper-large-v3-turbo-darija` (LoRA)  
LoRA adapter on `large-v3-turbo` (809M base).  Lighter download (base ~3 GB + adapter ~50 MB).

In [ ]:
result_c = transcribe_and_time(
    "anaszil/whisper-large-v3-turbo-darija", str(WAV_PATH),
    is_lora=True,
    lora_base="openai/whisper-large-v3-turbo",
)
pretty_print(result_c)

## 5. Comparison table

In [ ]:
from tabulate import tabulate

rows = []
for r in [result_a, result_b, result_c]:
    tag = r["model"].split("/")[-1]
    rows.append([
        tag,
        r.get("load_s", "N/A"),
        r.get("transcribe_s", "N/A"),
        "✔" if r.get("text") else r.get("error", "✘"),
        r["text"][:200] + ("..." if len(r["text"]) > 200 else ""),
    ])

print(tabulate(
    rows,
    headers=["Model", "Load (s)", "Transcribe (s)", "OK?", "First 200 chars"],
    tablefmt="grid",
))
print(f"\nAudio duration: {duration:.1f}s, length: {len(result_c.get('text','').split())} words (C)")

## 6. (Optional) Save results to file

In [ ]:
report = {
    "audio": str(WAV_PATH),
    "duration_s": round(duration, 1),
    "results": {
        r["model"]: {
            "load_s": r["load_s"],
            "transcribe_s": r["transcribe_s"],
            "text": r["text"],
            "error": r.get("error"),
        }
        for r in [result_a, result_b, result_c]
    },
}
out_path = OUT_DIR / "benchmark_results.json"
out_path.write_text(json.dumps(report, ensure_ascii=False, indent=2))
print(f"Saved to {out_path}")

## 7. (Optional) Compare with reference transcript

If you have a ground-truth transcript file, set `REF_PATH` below and the cell
will compute Word Error Rate (WER) for each model.

In [ ]:
REF_PATH = None  # set to a .txt file with the reference transcript

if REF_PATH and os.path.exists(REF_PATH):
    from jiwer import wer
    ref_text = Path(REF_PATH).read_text().strip()
    print("WER comparison (lower is better):")
    print(f"  {'Model':<50} {'WER':>8}")
    print(f"  {'-'*50} {'-'*8}")
    for r in [result_a, result_b, result_c]:
        hyp = r.get("text", "")
        if hyp:
            w = round(wer(ref_text, hyp), 2)
        else:
            w = "N/A"
        print(f"  {r['model']:<50} {w:>8}")
else:
    print("No REF_PATH set.  Skipping WER.")